In [38]:
from models import get_embeddings
from sentence_transformers import CrossEncoder
from pdf_chunk import text_spliter
from langchain_chroma import Chroma
from langchain_chroma import Chroma
from models import get_model , get_embeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableLambda

In [3]:

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 938.67it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
embedding_model = get_embeddings()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3613.30it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
chunks =  text_spliter

In [6]:
vectorstore=Chroma.from_documents(chunks ,embedding_model )
vectorstore_retreiver = vectorstore.as_retriever(search_kwargs={"k": 5})

In [7]:
query = "current university"

In [8]:
print("=== Vector Search Results ===")
vector_results = vectorstore_retreiver.invoke(query)
for i, doc in enumerate(vector_results):
    print(f"\n[{i+1}] {doc.page_content[:200]}")

=== Vector Search Results ===

[1] Role: MLOps Engineer
Company: Lyceum Infotech Private Limited
Duration: September 2022 to December 2024
Location: Pune, India

[2] Role: Assistant Professor
Institution: Guru Gobind Singh Foundation
Duration: May 2019 to August 2022
Location: Nashik, India

[3] Secondary Education:
SSC (Secondary School Certificate): 82%
HSC (Higher Secondary Certificate): 57%


COMPLETE TECHNICAL SKILLS

[4] CAREER OBJECTIVE

[5] He is currently open to ML Engineer, AI Engineer, and NLP Engineer roles.


In [9]:
pairs = []
for doc in vector_results:
    pairs.append([query, doc.page_content])

In [10]:
pairs

[['current university',
  'Role: MLOps Engineer\nCompany: Lyceum Infotech Private Limited\nDuration: September 2022 to December 2024\nLocation: Pune, India'],
 ['current university',
  'Role: Assistant Professor\nInstitution: Guru Gobind Singh Foundation\nDuration: May 2019 to August 2022\nLocation: Nashik, India'],
 ['current university',
  'Secondary Education:\nSSC (Secondary School Certificate): 82%\nHSC (Higher Secondary Certificate): 57%\n\n\nCOMPLETE TECHNICAL SKILLS'],
 ['current university', 'CAREER OBJECTIVE'],
 ['current university',
  'He is currently open to ML Engineer, AI Engineer, and NLP Engineer roles.']]

In [11]:
scores = cross_encoder.predict(pairs)
scores

array([-11.268473,  -9.076447, -10.95204 , -11.123005, -10.602231],
      dtype=float32)

In [12]:

scored_docs = zip(scores, vector_results)

In [13]:

reranked_document_cross_encoder = sorted(scored_docs, reverse=True)

In [14]:

reranked_document_cross_encoder

[(np.float32(-9.076447),
  Document(id='99ec9105-f3de-47b1-8a76-87ccf632f1d8', metadata={'source': 'my_data.txt'}, page_content='Role: Assistant Professor\nInstitution: Guru Gobind Singh Foundation\nDuration: May 2019 to August 2022\nLocation: Nashik, India')),
 (np.float32(-10.602231),
  Document(id='8be6a897-4881-4a85-8f33-ad8fc3a42476', metadata={'source': 'my_data.txt'}, page_content='He is currently open to ML Engineer, AI Engineer, and NLP Engineer roles.')),
 (np.float32(-10.95204),
  Document(id='0621193a-b918-4efc-a162-9f1c358e0994', metadata={'source': 'my_data.txt'}, page_content='Secondary Education:\nSSC (Secondary School Certificate): 82%\nHSC (Higher Secondary Certificate): 57%\n\n\nCOMPLETE TECHNICAL SKILLS')),
 (np.float32(-11.123005),
  Document(id='b9009967-2459-4fc3-b139-a8d85f9c2eb1', metadata={'source': 'my_data.txt'}, page_content='CAREER OBJECTIVE')),
 (np.float32(-11.268473),
  Document(id='720baeca-5610-46bd-b788-048f5f13c3d3', metadata={'source': 'my_data.txt

In [39]:
contents = [doc.page_content for score, doc in 
reranked_document_cross_encoder]

In [40]:
contents

['Role: Assistant Professor\nInstitution: Guru Gobind Singh Foundation\nDuration: May 2019 to August 2022\nLocation: Nashik, India',
 'He is currently open to ML Engineer, AI Engineer, and NLP Engineer roles.',
 'Secondary Education:\nSSC (Secondary School Certificate): 82%\nHSC (Higher Secondary Certificate): 57%\n\n\nCOMPLETE TECHNICAL SKILLS',
 'CAREER OBJECTIVE',
 'Role: MLOps Engineer\nCompany: Lyceum Infotech Private Limited\nDuration: September 2022 to December 2024\nLocation: Pune, India']

In [15]:
prompt = ChatPromptTemplate.from_template("""
Answer the question based on the context below.

Context: {context}
Question: {question}
""")

In [17]:
llm = get_model()

In [41]:
# First join contents into a single string
context_str = "\n\n".join(contents)

# Then pass directly
normal_chain = (
    {"context": RunnableLambda(lambda x: context_str), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [43]:
query = "what is current university of rohanta"

print(normal_chain.invoke(query))

The provided context does not mention the current university of Rohanta. It only mentions the institution where the person worked as an Assistant Professor, which is Guru Gobind Singh Foundation.
